### Merge Files

In [ ]:
#Old structure

import pandas as pd
from io import StringIO
from pathlib import Path

root = Path(r"D:\MUMBAI IGR\pahadi goregaon")
files = root.rglob("*.xls")

dfs = []

for file in files:
    try:
        with open(file, "r", encoding="utf-16") as f:
            df = pd.read_html(StringIO(f.read()))[0]

        df["_source_file"] = file.name
        dfs.append(df)

    except Exception as e:
        print("Failed:", file, e)

merged = pd.concat(dfs, ignore_index=True)

merged.to_excel(
    root / r"D:\chikhali\Chikhali\Chikhali_Merged_Data.xlsx",
    index=False
)

print("Rows:", len(merged))
print("Done")

In [2]:
#New Structure wise

import pandas as pd
from io import StringIO
from pathlib import Path

# Main folder containing both year folders
root = Path(r"D:\MUMBAI IGR\pahadi goregaon")

# Folders to merge
folders = [
    root / "2025 it 2",
    root / "2026"
]

dfs = []

for folder in folders:

    print(f"\n📂 Processing folder: {folder}")

    # Find all .xls files inside this folder
    files = folder.rglob("*.xls")

    for file in files:

        # Don't process the output file if it happens to be .xls
        if file.name == "Malad_Merged_Data.xlsx":
            continue

        try:
            print(f"   📄 Reading: {file.name}")

            # Read HTML-based .xls file
            with open(file, "r", encoding="utf-16") as f:
                html_content = f.read()

            tables = pd.read_html(StringIO(html_content))

            if not tables:
                print(f"   ⚠️ No table found: {file.name}")
                continue

            df = tables[0]

            # Add source information
            df["_source_file"] = file.name
            df["_source_folder"] = folder.name

            dfs.append(df)

            print(f"   ✅ Rows: {len(df)}")

        except Exception as e:
            print(f"   ❌ Failed: {file.name}")
            print(f"      Error: {e}")


# Check if anything was successfully loaded
if dfs:

    # Merge all files
    merged = pd.concat(dfs, ignore_index=True)

    # Output file
    output_file = root / "Malad_Merged_Data.xlsx"

    # Save
    merged.to_excel(output_file, index=False)

    print("\n" + "=" * 50)
    print("✅ MERGING COMPLETED")
    print("=" * 50)
    print(f"Total files merged : {len(dfs)}")
    print(f"Total rows         : {len(merged)}")
    print(f"Total columns      : {len(merged.columns)}")
    print(f"Output file        : {output_file}")

else:
    print("\n❌ No XLS files were successfully processed.")


📂 Processing folder: D:\MUMBAI IGR\pahadi goregaon\2025 it 2
   📄 Reading: pahadi_goregaon_1_2025.xls
   ✅ Rows: 7228
   📄 Reading: pahadi_goregaon_2_2025.xls
   ✅ Rows: 5931
   📄 Reading: pahadi_goregaon_3_2025.xls
   ✅ Rows: 2902
   📄 Reading: pahadi_goregaon_4_2025.xls
   ✅ Rows: 2746
   📄 Reading: pahadi_goregaon_5_2025.xls
   ✅ Rows: 4635
   📄 Reading: pahadi_goregaon_6_2025.xls
   ❌ Failed: pahadi_goregaon_6_2025.xls
      Error: no text parsed from document (line 0)
   📄 Reading: pahadi_goregaon_7_2025.xls
   ✅ Rows: 1267
   📄 Reading: pahadi_goregaon_8_2025.xls
   ✅ Rows: 1238
   📄 Reading: pahadi_goregaon_9_2025.xls
   ✅ Rows: 1724

📂 Processing folder: D:\MUMBAI IGR\pahadi goregaon\2026
   📄 Reading: pahadi_goregaon_1_2026.xls
   ✅ Rows: 4832
   📄 Reading: pahadi_goregaon_2_2026.xls
   ✅ Rows: 3894
   📄 Reading: pahadi_goregaon_3_2026.xls
   ✅ Rows: 1952
   📄 Reading: pahadi_goregaon_4_2026.xls
   ✅ Rows: 1963
   📄 Reading: pahadi_goregaon_5_2026.xls
   ✅ Rows: 2873
   📄 Rea

In [6]:
import pandas as pd
df = pd.read_excel("D:\IGR DATA PROCESSING\Kondhwa khurd\Code & Files\Kondhwa_Merged_Data.xlsx")

### Remove Duplicates From Merged File

In [3]:
import pandas as pd
from pathlib import Path

# ==========================================================
# INPUT & OUTPUT FILES
# ==========================================================
INPUT_FILE = r"D:\MUMBAI IGR\pahadi goregaon\Pahadi_goregaon_Merged_Data.xlsx"
OUTPUT_FILE = r"D:\MUMBAI IGR\pahadi goregaon\Pahadi_goregaon_Data.xlsx"

# ==========================================================
# COLUMNS TO KEEP
# ==========================================================
KEEP_COLUMNS = [
    'srocode',
    'internaldocumentnumber',
    'docno',
    'docname',
    'registrationdate',
    'sroname',
    'micrno',
    'bank_type',
    'party_code',
    'sellerparty',
    'purchaserparty',
    'propertydescription',
    'areaname',
    'consideration_amt',
    'marketvalue',
    'dateofexecution',
    'stampdutypaid',
    'registrationfees',
]

# ==========================================================
# DUPLICATE CHECK COLUMNS
# ==========================================================
DEDUP_SUBSET = [
    'docno',
    'docname',
    'registrationdate',
    'sroname',
    'propertydescription',
    'areaname',
    'consideration_amt',
    'marketvalue',
]

# ==========================================================
# LOAD & CLEAN FUNCTION
# ==========================================================
def load_and_clean(path: Path) -> pd.DataFrame:
    # Read Excel
    df = pd.read_excel(path)

    # Convert column names to lowercase
    df.columns = df.columns.str.lower().str.strip()

    # Keep only existing required columns
    existing_cols = [c for c in KEEP_COLUMNS if c in df.columns]
    df = df[existing_cols]

    # Remove rows where all kept columns are blank
    df = df.dropna(subset=existing_cols, how='all')

    # Clean propertydescription
    if 'propertydescription' in df.columns:
        df['propertydescription'] = (
            df['propertydescription']
            .astype(str)
            .str.strip()
            .replace({'': pd.NA, 'nan': pd.NA})
        )

        # Remove blank property descriptions
        df = df.dropna(subset=['propertydescription'])

        # Convert to Title Case
        df['propertydescription'] = df['propertydescription'].str.title()

    return df

# ==========================================================
# MAIN
# ==========================================================
def main():
    input_path = Path(INPUT_FILE)

    print("Reading Excel...")
    df = load_and_clean(input_path)

    print(f"Rows after cleaning: {len(df):,}")

    # Remove duplicates
    dedup_cols = [c for c in DEDUP_SUBSET if c in df.columns]

    before = len(df)
    df = df.drop_duplicates(subset=dedup_cols, keep='first')
    after = len(df)

    print(f"Duplicates removed : {before - after:,}")
    print(f"Final rows         : {after:,}")

    # Save output
    df.to_excel(OUTPUT_FILE, index=False)

    print(f"\nCleaned file saved to:\n{OUTPUT_FILE}")

# ==========================================================
# RUN
# ==========================================================
if __name__ == "__main__":
    main()

Reading Excel...
Rows after cleaning: 45,741
Duplicates removed : 19,604
Final rows         : 26,137

Cleaned file saved to:
D:\MUMBAI IGR\pahadi goregaon\Pahadi_goregaon_Data.xlsx


### Delete data After specific date

In [4]:
import pandas as pd

input_file = r"D:\MUMBAI IGR\pahadi goregaon\Pahadi_goregaon_Data.xlsx"
output_file = r"D:\MUMBAI IGR\pahadi goregaon\Pahadi_Goregaon_filtered_data.xlsx"

df = pd.read_excel(input_file)

# Convert registrationdate
df["registrationdate"] = pd.to_datetime(
    df["registrationdate"],
    format="%d/%m/%Y",
    errors="coerce"
)

# Delete dates before and including 05/09/2025
df = df[df["registrationdate"] >= pd.Timestamp("2025-10-30")]

# Save
df.to_excel(output_file, index=False)

print(f"Saved: {output_file}")

Saved: D:\MUMBAI IGR\pahadi goregaon\Pahadi_Goregaon_filtered_data.xlsx


### Divide Main File Into Parts For LLM

In [5]:
import pandas as pd

# ==========================================================
# Load Excel File
# ==========================================================
file_name = r"D:\MUMBAI IGR\pahadi goregaon\Pahadi_Goregaon_filtered_data.xlsx"

df = pd.read_excel(file_name)

# ==========================================================
# Split into 4 Parts
# ==========================================================
num_parts = 4
rows = len(df)

part_size = rows // num_parts

for i in range(num_parts):
    start = i * part_size

    # Last part gets the remaining rows
    if i == num_parts - 1:
        end = rows
    else:
        end = (i + 1) * part_size

    part = df.iloc[start:end]

    output_file = f"part_{i+1}.xlsx"
    part.to_excel(output_file, index=False)

    print(f"{output_file}: {len(part)} rows")

print(f"\nSuccessfully split {rows} rows into {num_parts} files.")

part_1.xlsx: 3433 rows
part_2.xlsx: 3433 rows
part_3.xlsx: 3433 rows
part_4.xlsx: 3434 rows

Successfully split 13733 rows into 4 files.


### Merge Output Proccessed Files

In [ ]:
#Note: All Files Must start with "part_"

import pandas as pd
import os

# === CONFIGURATION ===
input_folder = r"D:\Mohmadwadi\outpur processed"   # Folder containing batch files
output_file = r"D:\Mohmadwadi\outpur processed\Mohmadwadi_Final_Merged_File.xlsx"  # Output merged file

# === FIND ALL EXCEL FILES IN FOLDER ===
excel_files = [f for f in os.listdir(input_folder) if f.endswith('.xlsx') and f.startswith('part_')]

# === READ AND CONCATENATE ===
merged_df = pd.DataFrame()

for file in sorted(excel_files, key=lambda x: int(x.split('_')[1].split('.')[0])):  # ensures batch_1, batch_2, etc.
    file_path = os.path.join(input_folder, file)
    print(f"Reading {file_path}...")
    temp_df = pd.read_excel(file_path)
    merged_df = pd.concat([merged_df, temp_df], ignore_index=True)

# === SAVE FINAL MERGED FILE ===
merged_df.to_excel(output_file, index=False)
print(f"\n✅ Merged file created successfully: {output_file}")


In [ ]:
#Extra Merge code

import pandas as pd
from pathlib import Path

# ==========================================================
# INPUT / OUTPUT
# ==========================================================
ROOT_FOLDER = Path(r"D:\chikhali\Chikhali")
OUTPUT_FILE = ROOT_FOLDER / "Chikhali_Merged_Data.xlsx"

# ==========================================================
# FIND ALL FILES
# ==========================================================
excel_files = list(ROOT_FOLDER.rglob("*.xls"))
excel_files += list(ROOT_FOLDER.rglob("*.xlsx"))

print(f"Found {len(excel_files)} files")

all_data = []

# ==========================================================
# READ FILES
# ==========================================================
for file in excel_files:

    print(f"Reading: {file}")

    df = None

    # Try reading as Excel
    try:
        if file.suffix.lower() == ".xls":
            try:
                df = pd.read_excel(file, engine="xlrd")
            except:
                df = pd.read_excel(file)
        else:
            df = pd.read_excel(file, engine="openpyxl")

    except:
        pass

    # If not Excel, try reading as HTML
    if df is None:
        try:
            tables = pd.read_html(file)
            if len(tables) > 0:
                df = tables[0]
        except:
            pass

    # Skip unreadable files
    if df is None:
        print(f"Skipped: {file.name}")
        continue

    # Add source information
    df["Source_Folder"] = file.parent.name
    df["Source_File"] = file.name

    all_data.append(df)

# ==========================================================
# MERGE
# ==========================================================
if not all_data:
    print("No readable files found.")
else:
    merged_df = pd.concat(all_data, ignore_index=True)

    print("\n==============================")
    print("Merged Successfully")
    print("Files Read :", len(all_data))
    print("Rows       :", len(merged_df))
    print("Columns    :", len(merged_df.columns))
    print("==============================")

    merged_df.to_excel(OUTPUT_FILE, index=False)

    print(f"\nSaved to:\n{OUTPUT_FILE}")

### Delete Data As per Year(if required)

In [2]:
import pandas as pd

# File path
input_file = r"D:\Moshi\Pre Processing\Moshi\Moshi_Merged_Data_24-26.xlsx"
output_file = r"D:\Moshi\Pre Processing\Moshi\Moshi_Merged_Data_25-26.xlsx"

# Read Excel file
df = pd.read_excel(input_file)

# Remove rows where year = 2020
df = df[df['year'] != 2024]

# Save filtered data
df.to_excel(output_file, index=False)

print(f"Rows after removing year 2020: {len(df)}")
print(f"Saved file: {output_file}")

Rows after removing year 2020: 23836
Saved file: D:\Moshi\Pre Processing\Moshi\Moshi_Merged_Data_25-26.xlsx
